# Free GPU Qwen retrieval job
Your Mac exports a fixed source snapshot. This notebook runs the pinned Qwen3-Embedding-4B and Qwen3-Reranker-4B sequentially on one CUDA GPU, and produces numbers for local validation. No inference API, provider credential or review ledger is required. Results are candidate passages for human review.

Create a **private** Kaggle input dataset containing the exported `job.json` and the repository's `tools/qwen_kaggle_runner.py`. Upload this notebook to Kaggle. Set **GPU** (T4 x2 or P100 when your account offers it) and **Internet On**. Check your own available GPU quota. Keep the dataset and notebook private; they contain source text. Models use an Apache-2.0 license.

GPU memory fit and quality must be measured on the actual run. A missing GPU, overlong input, memory error or dependency mismatch stops execution. There is no paid or smaller-model fallback.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'transformers==4.51.3', 'tokenizers==0.21.1',
                'safetensors==0.5.3', 'huggingface-hub==0.30.2'], check=True)
# Keep Kaggle's preinstalled CUDA Torch; its actual version is recorded.
# If you have already imported Transformers, restart the session after installation.

## Set the two input paths and the trusted job checksum
Copy the job SHA-256 printed by the engine's local export. Do not infer it from an unknown job file. The model cache stays outside `/kaggle/working`, so saved notebook output contains results and checkpoints rather than model weights.

In [ ]:
import os
from pathlib import Path
os.environ['HF_HOME'] = '/kaggle/temp/huggingface'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
JOB_PATH = Path('/kaggle/input/YOUR_PRIVATE_DATASET/job.json')
RUNNER_PATH = Path('/kaggle/input/YOUR_PRIVATE_DATASET/qwen_kaggle_runner.py')
JOB_SHA256 = 'PASTE_TRUSTED_LOCAL_EXPORT_SHA256_HERE'
OUTPUT_PATH = Path('/kaggle/working/results.json')
CHECKPOINT_PATH = Path('/kaggle/working/checkpoint.json')

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('qwen_kaggle_runner', RUNNER_PATH)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
job = runner.validate_job(runner.read_artifact(JOB_PATH), JOB_SHA256)
import torch
assert torch.cuda.is_available(), 'Select a Kaggle CUDA GPU before running.'
print({'queries': len(job['queries']), 'representations': len(job['representations']),
       'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
       'embedding_revision': job['profile']['embedding_revision'],
       'reranker_revision': job['profile']['reranker_revision']})

## Run once, then stop the GPU session
The first run downloads both model revisions. It embeds sources and queries, releases the embedding model, then reranks the locally specified candidate pools. Complete input token counts are checked before inference; inputs are never truncated. A checkpoint is saved after each completed inference. Existing outputs are preserved: choose new output and checkpoint names for another run. Checkpoints are evidence of partial work and do not automatically resume or count as complete results.

In [ ]:
result = runner.run_job(JOB_PATH, job_sha256=JOB_SHA256,
                        output_path=OUTPUT_PATH, checkpoint_path=CHECKPOINT_PATH)
print(runner.canonical(result))

## Return results to your Mac
Save the notebook version with outputs. Download `results.json` from `/kaggle/working` and keep the printed `results_sha256` with it. Import it using the original local job file, trusted job checksum and returned results checksum. The engine recomputes candidate pools and ranks, checks source versions and screening eligibility, and writes an immutable receipt. A changed source or screening decision requires a fresh export. Stop the Kaggle session when finished to conserve your account's quota.

The human pilot remains pending until the frozen medical retrieval evaluation and actual GPU run pass the documented acceptance criteria.